# L3 — Gateway Tools (Order, Refund & Web Search)

## What you'll learn

- How agents reach their tools through one AgentCore Gateway
- How to add the Order and Refund tools to the Gateway
- How to connect the web-search tool of your choice: AgentCore Web Search, Brave Search or Perplexity Search
- How to confirm where agent traces will go: CloudWatch, Datadog or CloudZero depending on configuration

## Overview

The specialist agents never call their tools directly. Every tool call goes through the **AgentCore Gateway**, which sends it to the right tool. This notebook adds three tools to the Gateway:

- **Order tool.** Looks up customer orders in DynamoDB. Used by the Order Agent.
- **Refund tools.** Answer return-policy questions from the vector store you set up in L1 (Bedrock Knowledge Base or Couchbase), check refund eligibility, and process refunds. Used by the Refund Agent.
- **Web-search tool.** Searches the web for product reviews and specs. Used by the Research Agent. You choose the provider in the Partner Selection UI.

The Gateway and its permissions are already set up by the workshop, so this notebook only adds the tools.

## Architecture

![Orchestration layer: the AgentCore Harness with its memory discovers the Order, Refund and Research agents through the Registry; the agents call tools through the AgentCore Gateway, which reaches the order tools and DynamoDB, the refund tools and the knowledge base (Bedrock or Couchbase), and one web-search provider.](images/l3-orchestration-architecture.png)

This notebook adds the Gateway targets and the tools behind them, the two right-hand columns of the diagram. The next notebooks add the agents and the orchestrator.

## Steps

0. Install the packages
1. Load the workshop configuration
2. Add the Order tool
3. Add the Refund tools
4. Connect the web-search tool (code from the Partner Selection UI)
5. Save the tool IDs
6. Configure Agent Observability Provider (code from the Partner Selection UI)

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- The Partner Selection UI from L0 ([`1_partner_selection.ipynb`](../l0-partner-selection/1_partner_selection.ipynb)) is running.
- L1 ([`1_knowledge_base.ipynb`](../l1-data-knowledge/1_knowledge_base.ipynb)) and L2 ([`1_pluggable_inference_layer.ipynb`](../l2-inference/1_pluggable_inference_layer.ipynb)) are complete.
- Run this notebook before the other L3 notebooks.

## Step 0: Install the packages

Install the AWS SDK. Run this once, at the start.

In [ ]:
%pip install --quiet --upgrade boto3

## Step 1: Load the workshop configuration

Read the settings the workshop saved earlier: the Gateway, the orders table, the tools' permissions, and the vector store you set up in L1 (Bedrock Knowledge Base or Couchbase). If L1 hasn't been run, the cell stops with a message that says so.

In [ ]:
import boto3
import os

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")  # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)


def get_ssm(name):
    return ssm.get_parameter(Name=f"{SSM_PREFIX}/{name}")["Parameter"]["Value"]


GATEWAY_ID = get_ssm("gateway_id")
ORDERS_TABLE = get_ssm("dynamodb_orders_table")
ORDER_TOOLS_ROLE_ARN = get_ssm("order_tools_role_arn")
REFUND_TOOLS_ROLE_ARN = get_ssm("refund_tools_role_arn")
# The knowledge store L1 set up (kb_provider): Bedrock Knowledge Base or Couchbase.
from helpers.gateway_tools import refund_kb_settings
REFUND_KB = refund_kb_settings(REGION)

print(f"Region:        {REGION}")
print(f"Gateway:       {GATEWAY_ID}")
print(f"Orders table:  {ORDERS_TABLE}")
print(f"Knowledge:     {REFUND_KB['provider']}")

### The tools

Each tool is a Lambda function plus a description of what it offers. The code and descriptions are in the [`tools/`](tools/) folder, and the workshop has already created each function's permissions.

In [ ]:
TOOLS = {
    "order": {
        "label": "Order tool",
        "lambda_name": "anycompany_order_tools",
        "source": "tools/order_tools.py",
        "schema": "tools/order_tools_schema.json",
        "target_name": "order-tools",
        "description": "Order lookup tools (order details, customer orders) backed by Lambda + DynamoDB",
        "role_arn": ORDER_TOOLS_ROLE_ARN,   # read-only access to the CustomerOrders table
        "env": {"ORDERS_TABLE": ORDERS_TABLE, "AGENT_REGION": REGION},
    },
    "refund": {
        "label": "Refund tools",
        "lambda_name": "anycompany_refund_tools",
        "source": "tools/refund_tools.py",
        "schema": "tools/refund_tools_schema.json",
        "target_name": "refund-tools",
        "description": "Refund tools (policy, eligibility, processing) backed by Lambda",
        "role_arn": REFUND_TOOLS_ROLE_ARN,  # Knowledge Base retrieval and the CustomerOrders table
        "env": {**REFUND_KB["env"], "AGENT_REGION": REGION},
        "packages": REFUND_KB["packages"],      # Couchbase: the Couchbase SDK
        "vpc_config": REFUND_KB["vpc_config"],  # Couchbase: the egress subnet on the Capella allowlist
    },
}

from helpers.gateway_tools import deploy_tool_lambda, add_gateway_target

## Step 2: Add the Order tool

Create the Order tool's Lambda function and add it to the Gateway, so the Order Agent can look up orders. The helper code is in [`helpers/gateway_tools.py`](helpers/gateway_tools.py).

In [ ]:
ORDER_LAMBDA_ARN = deploy_tool_lambda(TOOLS["order"], REGION)
ORDER_TARGET_ID = add_gateway_target(TOOLS["order"], ORDER_LAMBDA_ARN, GATEWAY_ID, REGION)

## Step 3: Add the Refund tools

Do the same for the Refund Agent's three tools: answering policy questions, checking eligibility, and processing a refund. If you chose Couchbase in L1, the notebook also connects this tool to your Couchbase cluster.

In [ ]:
REFUND_LAMBDA_ARN = deploy_tool_lambda(TOOLS["refund"], REGION)
REFUND_TARGET_ID = add_gateway_target(TOOLS["refund"], REFUND_LAMBDA_ARN, GATEWAY_ID, REGION)

## Step 4: Connect the web-search tool

The Research Agent searches the web for product reviews, specs and comparisons. You choose the search provider in the Partner Selection UI. Each one plugs into the same Gateway, so the agent works the same whichever you pick.

- **AgentCore Web Search (AWS):** a managed connector. No key needed.
- **Brave Search:** the Brave search server runs in your account (its image is already built). Needs your Brave API key.
- **Perplexity Search:** Perplexity's hosted search server. Needs your Perplexity API key.

Answers that use web results must cite their sources. The Research Agent is instructed to do this for every provider.

### 4.1 Connect the provider you picked

Paste the code for your provider from the Partner Selection UI (Orchestration → Tools) into the empty cell below, then run it.

<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Orchestration → Tools</b>.</li>
<li>Pick a web-search provider. For Brave Search or Perplexity Search, save its API key.</li><li>Expand <b>📋 1 · Connect the web-search tool</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "1 · Connect the web-search tool" snippet from the Partner Selection UI (Orchestration → Tools) here, then run.



In [ ]:
from helpers import websearch as ws

if "WEBSEARCH" not in globals():
    raise RuntimeError("No web-search tool connected. Paste the Connect snippet from the Partner "
                       "Selection UI (Orchestration -> Tools) into the cell above and run it.")
print("Active provider:", ws.active(REGION))

### 4.2 Test it

Search once through the Gateway, the same way the Research Agent will. If results come back, the tool is working.

To switch providers later, paste a different provider's code into 4.1, run it, then re-run this cell. The Research Agent picks up the change within about a minute.

In [ ]:
_ = ws.smoke_test(REGION, query="noise cancelling headphones reviews", max_results=3)

## Step 5: Save the tool IDs

Save the ID of each tool's function and Gateway target, so the later notebooks can find them.

In [ ]:
# Save tool IDs to SSM so the agent notebooks (2-5) and L4/L5 can read them.
params = {
    f"{SSM_PREFIX}/order_lambda_arn": ORDER_LAMBDA_ARN,
    f"{SSM_PREFIX}/order_gateway_target_id": ORDER_TARGET_ID,
    f"{SSM_PREFIX}/refund_lambda_arn": REFUND_LAMBDA_ARN,
    f"{SSM_PREFIX}/refund_gateway_target_id": REFUND_TARGET_ID,
}
# Web-search tool (Step 4). WEBSEARCH is set by the Connect cell you pasted in 4.1.
if "WEBSEARCH" in globals():
    params[f"{SSM_PREFIX}/websearch_gateway_target_id"] = WEBSEARCH["target_id"]
else:
    print("  (web-search tool not connected in this session - run Step 4.1 to record its target ID)")
for name, value in params.items():
    ssm.put_parameter(Name=name, Value=value, Type="SecureString", Overwrite=True)
    print(f"  {name} = {value}")
print("\nDone.")


## Step 6: Configure Agent Observability Provider

Every agent sends its traces to the observability provider you pick in the Partner Selection UI (**Orchestration → Agent Observability**): **Amazon CloudWatch** (the default), **Datadog** or **CloudZero**. With CloudZero, traces stay on CloudWatch and the Order, Refund and Research agents also send each model call's span (tokens and cost, no prompt text) to CloudZero; the orchestrator is not sent. Logs always go to CloudWatch. If you don't pick one, the agents use CloudWatch.

Paste the **1 · Confirm the observability provider** code from the Partner Selection UI into the empty cell below, then run it. It checks your choice and, for Datadog or CloudZero, that your credentials are saved, so a problem shows up now instead of when you deploy the agents. To switch later, change it in the UI, paste the other provider's code here, re-run it, then redeploy the agents.

<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Orchestration → Agent Observability</b>.</li>
<li>Click <b>CloudWatch</b>; or click <b>Datadog</b>, enter your keys and site, and select <b>Save to SSM</b>; or click <b>CloudZero</b>, enter your API key and username, and select <b>Save to SSM</b>.</li><li>Expand <b>📋 1 · Confirm the observability provider</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "1 · Confirm the observability provider" snippet from the Partner Selection UI (Orchestration → Agent Observability) here, then run.



## Summary

This notebook added three tools to the Gateway: the **Order tool** and the **Refund tools**, each with the Lambda function behind it, and the **web-search tool** you chose. Their IDs and your observability choice are saved for the later notebooks.

**Next:** [`2_specialist_agents.ipynb`](2_specialist_agents.ipynb). Deploy the Order, Refund and Research agents that use these tools.

---
## Cleanup
Deletes the tools this notebook created: the two Gateway targets and their Lambdas, the web-search tool, and their SSM parameters. Your saved API keys are kept. The Gateway, Registry, and IAM roles belong to the workshop stack and are removed when it's deleted.

Run this **after** the agent notebooks' cleanup. Uncomment the cell to run it.

In [ ]:
## === CLEANUP: delete the tools created in this notebook ===
## Uncomment and run ONLY after you are done with ALL L3 notebooks, and AFTER
## the agent notebooks' cleanup has removed their registry records.

# import boto3, os
# REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
# SSM_PREFIX = "/anycompany/agentcore"
# lambda_client = boto3.client("lambda", region_name=REGION)
# ssm = boto3.client("ssm", region_name=REGION)
# agentcore_control = boto3.client("bedrock-agentcore-control", region_name=REGION)
#
# def _ssm_get(key):
#     try: return ssm.get_parameter(Name=f"{SSM_PREFIX}/{key}")["Parameter"]["Value"]
#     except ssm.exceptions.ParameterNotFound: return None
#
# JWT_GW = _ssm_get("gateway_id")
# HARNESS_GW = (_ssm_get("harness_gateway_arn") or "").split("/")[-1]
#
# # The tool Lambdas' IAM roles are owned by the CloudFormation stack (not deleted here).
# TOOLS = [
#     ("anycompany_order_tools",    "order-tools",  JWT_GW),
#     ("anycompany_refund_tools",   "refund-tools", JWT_GW),
# ]
# for lambda_name, target_name, gw in TOOLS:
#     if gw:
#         try:
#             tg = agentcore_control.list_gateway_targets(gatewayIdentifier=gw)
#             for t in tg.get("items", tg.get("targets", [])):
#                 if t.get("name") == target_name:
#                     agentcore_control.delete_gateway_target(gatewayIdentifier=gw, targetId=t["targetId"])
#                     print(f"Deleted gateway target: {target_name}")
#         except Exception as e: print(f"target {target_name}: {e}")
#     try:
#         lambda_client.delete_function(FunctionName=lambda_name); print(f"Deleted Lambda: {lambda_name}")
#     except lambda_client.exceptions.ResourceNotFoundException: pass
#     except Exception as e: print(f"lambda {lambda_name}: {e}")
#
# for param in ["order_lambda_arn","order_gateway_target_id","refund_lambda_arn","refund_gateway_target_id"]:
#     try: ssm.delete_parameter(Name=f"{SSM_PREFIX}/{param}"); print(f"Deleted SSM: {param}")
#     except ssm.exceptions.ParameterNotFound: pass
#
# # Web-search tool: every web-search target, the Brave runtime, the Perplexity
# # credential provider, the Brave secret and the websearch_* SSM parameters
# # (including websearch_gateway_target_id).
# from helpers import websearch as ws
# ws.disconnect(REGION)
#
# print("\n All L3 tool resources cleaned up.")